In [ ]:
import sys
import torch
from pathlib import Path
import json
from tqdm import tqdm

sys.path.append(str(Path(__file__).parent.parent))

from tokenizer import ProteinTokenizer
from phylogen.model import PhyloGen
from phylogen.dataset import ProteomeDataset

# === PATHS ===
BASE_DIR = Path(__file__).parent.parent if "__file__" in globals() else Path.cwd().parent
csv_path       = BASE_DIR / "data" / "ecoli_processed_pairs" / "ecoli_pairs_concat.csv"
phylo_pkl_path = BASE_DIR / "data" / "gtdb_data" / "ecoli_phylo_distances.pkl"
tokenizer_path = BASE_DIR / "tokenizer" / "tokenizer.json"
ckpt_path      = BASE_DIR / "checkpoints_finetune" / "finetune_epoch_3.pt"

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer
print("Loading tokenizer...")
tokenizer = ProteinTokenizer.load(str(tokenizer_path))

# Load model
print("Loading model architecture...")
model = PhyloGen(
    vocab_size=tokenizer.vocab_size,
    tokenizer=tokenizer,
    embed_dim=256,
    num_heads=8,
    num_layers=6,
    max_seq_len=2048,
)

print("Loading finetune checkpoint...")
ckpt = torch.load(ckpt_path, map_location="cpu")
model_state = ckpt.get('model', ckpt)
model.load_state_dict(model_state, strict=False)
model = model.to(device)
model.eval()
print("Model loaded successfully\n")

# Load small dataset subset for testing (fast)
print("Loading small dataset subset for inference testing...")
chunk_cache_pkl = BASE_DIR / "cache" / "ecoli_pairs_concat_mode_finetune_chunk1024_overlap512_mutated_only_True_start0_maxall.chunks.pkl"
dataset = ProteomeDataset(
    csv_path=str(csv_path),
    tokenizer=tokenizer,
    chunk_size=1024,
    overlap=512,
    phylo_pkl=str(phylo_pkl_path),
    mode="finetune",
    max_samples=5,
    use_mutated_only=True,
    force_recompute=False,
    chunk_cache_pkl=str(chunk_cache_pkl) if chunk_cache_pkl.exists() else None,
)
print(f"Dataset ready: {len(dataset)} chunks from {len(dataset.df)} genomes\n")


In [ ]:
# constrained_generate.py
import torch
from tqdm import tqdm

@torch.no_grad()
def generate_constrained(
    model,
    tokenizer,
    unmutated_proteome: str,
    mutation_positions: list[int],   # relative to mutated start
    mutation_targets: list[str],     # target AAs like ['I', 'L', 'N']
    target_length: int = None,
    max_new_tokens_per_window: int = 2000,
    overlap_tokens: int = 384,
    temperature: float = 0.0,
    device=None,
    max_prompt_prefix: int = 12000,
):
    if device is None:
        device = next(model.parameters()).device

    if target_length is None:
        target_length = len(unmutated_proteome) + 20000

    cond = ["[SPECIES_ECOLI]", "[CIPRO]", "[RESISTANT]"]
    cond_ids = torch.tensor([tokenizer.vocab[c] for c in cond], dtype=torch.long, device=device)
    bos_tensor = torch.tensor([tokenizer.bos_token_id], device=device)
    sep_tensor = torch.tensor([tokenizer.vocab["[SEP]"]], device=device)

    print("Encoding unmutated proteome...")
    unmut_ids = torch.tensor(
        tokenizer.encode_fast(unmutated_proteome, add_special_tokens=False),
        device=device
    )

    # Absolute positions in final generated sequence (after [SEP])
    continuation_start_offset = len(cond_ids) + 1 + 1  # BOS + cond + [SEP]
    edit_abs_positions = {continuation_start_offset + p for p in mutation_positions}

    full_generated_after_sep = []
    forced_at_positions = []   # to track where we forced edits

    current_pos = max(0, len(unmut_ids) - max_prompt_prefix)

    print(f"Starting generation from position {current_pos:,} (target length ~{target_length:,})")

    pbar = tqdm(total=target_length, desc="Generating", unit="tokens")

    while len(full_generated_after_sep) < target_length:
        window_start = max(0, current_pos - overlap_tokens)
        window_unmut = unmut_ids[window_start:current_pos]

        prompt = torch.cat([bos_tensor, cond_ids, window_unmut, sep_tensor]).unsqueeze(0)

        generated = prompt.clone()
        tokens_this_window = 0

        while tokens_this_window < max_new_tokens_per_window:
            if generated.shape[1] >= 2048:
                break

            context = generated[:, -2048:]
            out = model(context, phylo_dists=None)   # adjust if your model needs phylo
            logits = out["logits"][:, -1, :]

            if temperature > 0:
                probs = torch.softmax(logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = logits.argmax(dim=-1, keepdim=True)

            next_global_pos = current_pos + tokens_this_window

            # === FORCE MUTATION if at known position ===
            if next_global_pos in edit_abs_positions:
                idx = list(edit_abs_positions).index(next_global_pos)
                target_aa = mutation_targets[idx]
                target_id = tokenizer.vocab.get(target_aa, tokenizer.unk_token_id)
                next_token = torch.tensor([[target_id]], device=device)
                forced_at_positions.append(next_global_pos)
                print(f"  Forced mutation at {next_global_pos:,}: {target_aa}")

            # === FORCE COPY from unmutated for non-mutation positions ===
            else:
                rel_pos = next_global_pos - continuation_start_offset
                if 0 <= rel_pos < len(unmut_ids):
                    copy_id = unmut_ids[rel_pos]
                    next_token = torch.tensor([[copy_id]], device=device)

            generated = torch.cat([generated, next_token], dim=1)
            full_generated_after_sep.append(next_token.item())

            tokens_this_window += 1
            pbar.update(1)

            if len(full_generated_after_sep) >= target_length:
                break

        current_pos += max_new_tokens_per_window - overlap_tokens

    pbar.close()

    generated_text = tokenizer.decode(full_generated_after_sep, skip_special=True)

    return generated_text, full_generated_after_sep, forced_at_positions

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Pick one genome to test
idx = 0
row = dataset.df.iloc[idx]
unmut = row['unmutated_proteome']
mut   = row['mutated_proteome']

# Get real mutations
changes = [(j, unmut[j], mut[j]) for j in range(min(len(unmut), len(mut))) if unmut[j] != mut[j]]
mutation_pos = [p for p, _, _ in changes]
mutation_aas = [new for _, _, new in changes]

print(f"\nTesting genome {row['genome_id']}")
print(f"Known mutations: positions={mutation_pos}, targets={mutation_aas}")

orig_unmut_len = len(unmut)
orig_mut_len   = len(mut)

generated_cont, generated_tokens, forced_positions = generate_constrained(
    model=model,
    tokenizer=tokenizer,
    unmutated_proteome=unmut,
    mutation_positions=mutation_pos,
    mutation_targets=mutation_aas,
    target_length=len(mut) + 10000,
    max_new_tokens_per_window=2000,
    overlap_tokens=384,
    temperature=0.0
)

gen_len = len(generated_cont)

mutation_scaled = [p / orig_mut_len for p in mutation_pos]

fig, ax = plt.subplots(figsize=(14, 6))

ax.bar(0, orig_unmut_len, color='lightblue', width=0.3, label='Unmutated (input)')
ax.bar(1, orig_mut_len,   color='salmon',    width=0.3, label='Target mutated')
ax.bar(2, gen_len,        color='lightgreen', width=0.3, label='Generated')

ax.scatter([1] * len(mutation_pos), [orig_mut_len * p for p in mutation_scaled],
           c='lime', s=120, edgecolor='darkgreen', label='Known mutation positions', zorder=10)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Unmutated', 'Target Mutated', 'Generated'])
ax.set_ylabel('Length (AA tokens)')
ax.set_title('Full Sequence Comparison + Mutation Positions')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("full_sequence_comparison.png", dpi=150)
plt.show()